In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from pathlib import Path
from scipy.ndimage import map_coordinates as _map_coords
from statsmodels.tsa.seasonal import STL
import time, gc, warnings, math, copy
warnings.filterwarnings('ignore')

**Config**

In [2]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props      = torch.cuda.get_device_properties(0)
    DTYPE      = torch.bfloat16 if props.major >= 8 else torch.float16
    USE_AMP    = True
    USE_SCALER = (DTYPE == torch.float16)
    torch.backends.cudnn.benchmark     = True
    torch.backends.cudnn.deterministic = False
    print(f"GPU  : {torch.cuda.get_device_name()}")
    print(f"VRAM : {props.total_memory / 1024**3:.1f} GB")
    print(f"dtype: {DTYPE}")
else:
    DTYPE      = torch.float32
    USE_AMP    = False
    USE_SCALER = False
    print("WARNING: CPU only")
 
if Path('/kaggle/input').exists():
    ROOT   = Path('/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2')
    OUTPUT = Path('/kaggle/working')
    ENV    = 'kaggle'

# ye configure kr lena @anas
elif Path('/content').exists():
    ROOT   = Path('')
    OUTPUT = Path('/content/phase2_out')
    ENV    = 'colab'
else:
    ROOT   = Path(r'')
    OUTPUT = Path(r'')
    ENV    = 'local'
 
RAW     = ROOT / 'raw'
TEST_IN = ROOT / 'test_in'
CKPT    = OUTPUT / 'checkpoints'
OUTPUT.mkdir(exist_ok=True, parents=True)
CKPT.mkdir(exist_ok=True, parents=True)

# Data config
MONTHS       = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
MET_VARS     = ['q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain']
EMIS_VARS    = ['PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']
ALL_FEATURES = ['cpm25'] + MET_VARS + EMIS_VARS
 
# HIST_LEN=10 only. No future met.
HIST_LEN  = 10
HORIZON   = 16
# Model input uses HIST_LEN only (not WINDOW=26)
VAL_HOURS = 120
CKPT_EVERY = 10
 
# channel count
# Per timestep (HIST_LEN=10 steps)=
#   cpm25(1) + met(8) + emis(7) + wind_speed(1) + wind_sin(1) + wind_cos(1) + divergence(1) + persistence(1) + diurnal_sin(1) + diurnal_cos(1)  + lat(1) + lon(1) = 26 channels
C_IN = 26
 
# Model config 
BASE_CH  = 64
C_ENC    = 128
N_BLOCKS = 4
N_HEADS  = 8
DROPOUT  = 0.1
PAD_H, PAD_W = 4, 4
H_ENC = (140 + 2*PAD_H) // 4   # 37
W_ENC = (124 + 2*PAD_W) // 4   # 33
 
# Training config 
BS         = 4
ACCUM      = 4       # effective BS = 16
LR_PEAK    = 2e-4
WD         = 1e-4
WARMUP_EP  = 5
EPOCHS     = 80
PATIENCE   = 20
EMA_DECAY  = 0.9995
 
# Episode weighting
EPISODE_WEIGHT = 3.0   # loss multiplier for episodic grid points
EPISODE_PCT    = 90    # percentile threshold for episode detection during training
 
print(f"\nEnv    : {ENV}")
print(f"C_IN   : {C_IN}  (HIST_LEN={HIST_LEN}, no future met leak)")
print(f"Epochs : {EPOCHS}, Patience: {PATIENCE}")
assert RAW.exists(), f"Data not found: {RAW}"
print("Data path OK")

GPU  : Tesla T4
VRAM : 14.6 GB
dtype: torch.float16

Env    : kaggle
C_IN   : 26  (HIST_LEN=10, no future met leak)
Epochs : 80, Patience: 20
Data path OK


**Data Loading**

In [3]:
print("\nLoading raw data")
all_data = {}
for month in MONTHS:
    t0 = time.time()
    all_data[month] = {}
    for feat in ALL_FEATURES:
        all_data[month][feat] = np.load(RAW / month / f"{feat}.npy")
    T = all_data[month]['cpm25'].shape[0]
    print(f"  {month}: {T} hrs  ({time.time()-t0:.1f}s)")
 
lat_lon = np.load(RAW / 'lat_long.npy')   # (140, 124, 2)
print(f"  lat_lon: {lat_lon.shape}")


Loading raw data
  APRIL_16: 715 hrs  (5.8s)
  JULY_16: 739 hrs  (6.4s)
  OCT_16: 739 hrs  (6.4s)
  DEC_16: 739 hrs  (6.3s)
  lat_lon: (140, 124, 2)


In [4]:
#Normalizer 
class Normalizer:
    def __init__(self):
        self.stats = {}
 
    def fit(self, data_dict, months):
        print("Fitting normalizer")
        for feat in ALL_FEATURES:
            s, sq, n = 0.0, 0.0, 0
            for m in months:
                a = data_dict[m][feat].astype(np.float64)
                s += a.sum(); sq += (a**2).sum(); n += a.size
            mean = s / n
            std  = float(np.sqrt(max(sq/n - mean**2, 0)))
            self.stats[feat] = {'mean': float(mean), 'std': max(std, 1e-8)}
 
        # Wind-derived
        s, sq, n = 0.0, 0.0, 0
        for m in months:
            u   = data_dict[m]['u10'].astype(np.float64)
            v   = data_dict[m]['v10'].astype(np.float64)
            arr = np.sqrt(u**2 + v**2)
            s += arr.sum(); sq += (arr**2).sum(); n += arr.size
        mean = s / n
        std  = float(np.sqrt(max(sq/n - mean**2, 0)))
        self.stats['wind_speed'] = {'mean': float(mean), 'std': max(std, 1e-8)}
        self.stats['wind_sin']   = {'mean': 0.0, 'std': 1.0}
        self.stats['wind_cos']   = {'mean': 0.0, 'std': 1.0}
 
        # Episode threshold (90th pct of cpm25 for episode-aware loss)
        all_pm25 = np.concatenate([
            data_dict[m]['cpm25'].ravel() for m in months
        ])
        self.episode_threshold = float(np.percentile(all_pm25, EPISODE_PCT))
        print(f"  cpm25 mean={self.stats['cpm25']['mean']:.2f} "
              f"std={self.stats['cpm25']['std']:.2f}")
        print(f"  episode threshold (p{EPISODE_PCT}): "
              f"{self.episode_threshold:.2f} µg/m³")
 
    def norm(self, arr, feat):
        s = self.stats[feat]
        return (arr - s['mean']) / s['std']
 
    def denorm_pm25(self, arr):
        s = self.stats['cpm25']
        return arr * s['std'] + s['mean']
 
    @property
    def episode_threshold_norm(self):
        s = self.stats['cpm25']
        return (self.episode_threshold - s['mean']) / s['std']
 
 
normalizer = Normalizer()
normalizer.fit(all_data, MONTHS)
 
U10_MEAN = np.float32(normalizer.stats['u10']['mean'])
U10_STD  = np.float32(normalizer.stats['u10']['std'])
V10_MEAN = np.float32(normalizer.stats['v10']['mean'])
V10_STD  = np.float32(normalizer.stats['v10']['std'])

Fitting normalizer
  cpm25 mean=33.61 std=52.28
  episode threshold (p90): 83.13 µg/m³


In [5]:
#pre-normalize
print("\nPre-normalizing")
all_norm = {}
for month in MONTHS:
    nd = {}
    for feat in ALL_FEATURES:
        nd[feat] = normalizer.norm(
            all_data[month][feat], feat
        ).astype(np.float32)
    u   = all_data[month]['u10'].astype(np.float64)
    v   = all_data[month]['v10'].astype(np.float64)
    ws  = np.sqrt(u**2 + v**2)
    wd  = np.arctan2(v, u + 1e-8)
    nd['wind_speed'] = normalizer.norm(ws, 'wind_speed').astype(np.float32)
    nd['wind_sin']   = np.sin(wd).astype(np.float32)
    nd['wind_cos']   = np.cos(wd).astype(np.float32)
    del u, v, ws, wd
    all_norm[month] = nd
    print(f"  {month} done")
 
del all_data
gc.collect()
 
lat_norm = ((lat_lon[..., 0] - lat_lon[..., 0].mean()) /
            (lat_lon[..., 0].std() + 1e-8)).astype(np.float32)
lon_norm = ((lat_lon[..., 1] - lat_lon[..., 1].mean()) /
            (lat_lon[..., 1].std() + 1e-8)).astype(np.float32)
STATIC_CH = np.stack([lat_norm, lon_norm], axis=0)  # (2, 140, 124)
print(f"  STATIC_CH: {STATIC_CH.shape}")


Pre-normalizing
  APRIL_16 done
  JULY_16 done
  OCT_16 done
  DEC_16 done
  STATIC_CH: (2, 140, 124)


In [6]:
#phys feats.
def warp_pm25(pm25, u_norm, v_norm, dx=25000.0, dy=25000.0, dt=3600.0):
    H, W     = pm25.shape
    u_ms     = u_norm * U10_STD + U10_MEAN
    v_ms     = v_norm * V10_STD + V10_MEAN
    disp_col = (u_ms  * dt / dx)
    disp_row = (-v_ms * dt / dy)
    base_row = np.arange(H, dtype=np.float32)[:, None]
    base_col = np.arange(W, dtype=np.float32)[None, :]
    src_row  = np.clip(base_row + disp_row, 0, H - 1).ravel()
    src_col  = np.clip(base_col + disp_col, 0, W - 1).ravel()
    warped   = _map_coords(pm25, [src_row, src_col], order=1,
                           mode='nearest', prefilter=False)
    return warped.reshape(H, W).astype(np.float32)
 
 
def compute_wind_warp_hist(pm25_history, u_seq, v_seq):
    """Warp over HIST_LEN only, no future met leak."""
    warped = np.empty_like(pm25_history)
    warped[0] = pm25_history[0]
    for ts in range(1, HIST_LEN):
        warped[ts] = warp_pm25(pm25_history[ts-1], u_seq[ts-1], v_seq[ts-1])
    return warped
 
 
def compute_divergence_hist(u_seq, v_seq):
    """Divergence over HIST_LEN only."""
    du_dx           = np.empty_like(u_seq)
    du_dx[:, :, 1:-1] = (u_seq[:, :, 2:] - u_seq[:, :, :-2]) * 0.5
    du_dx[:, :, 0]    = u_seq[:, :, 1]  - u_seq[:, :, 0]
    du_dx[:, :, -1]   = u_seq[:, :, -1] - u_seq[:, :, -2]
    dv_dy           = np.empty_like(v_seq)
    dv_dy[:, 1:-1, :] = (v_seq[:, 2:, :] - v_seq[:, :-2, :]) * 0.5
    dv_dy[:, 0, :]    = v_seq[:, 1, :]  - v_seq[:, 0, :]
    dv_dy[:, -1, :]   = v_seq[:, -1, :] - v_seq[:, -2, :]
    return (du_dx + dv_dy).astype(np.float32)
 
 
# Diurnal encoding for HIST_LEN steps
_ts_arr     = np.arange(HIST_LEN, dtype=np.float32)
DIURNAL_SIN = np.sin(2 * np.pi * _ts_arr / 24).astype(np.float32)
DIURNAL_COS = np.cos(2 * np.pi * _ts_arr / 24).astype(np.float32)
 
print("Physics features defined.")

Physics features defined.


In [7]:
#hist_len only 10  feature extraction
def extract_phase2(nd, t):
    """
    Extract HIST_LEN=10 timestep input.
    Output shape: (HIST_LEN, C_IN, 140, 124)
    """
    H, W = 140, 124
 
    pm25_hist = nd['cpm25'][t:t+HIST_LEN]          # (10, H, W)
    u_seq     = nd['u10'][t:t+HIST_LEN]
    v_seq     = nd['v10'][t:t+HIST_LEN]
 
    warp_seq  = compute_wind_warp_hist(pm25_hist, u_seq, v_seq)  # (10, H, W)
    div_seq   = compute_divergence_hist(u_seq, v_seq)            # (10, H, W)
    persistence = pm25_hist[-1]                                   # (H, W)
 
    met  = np.stack([nd[v][t:t+HIST_LEN] for v in MET_VARS],  axis=1)  # (10,8,H,W)
    emis = np.stack([nd[v][t:t+HIST_LEN] for v in EMIS_VARS], axis=1)  # (10,7,H,W)
    wind = np.stack([nd['wind_speed'][t:t+HIST_LEN],
                     nd['wind_sin'][t:t+HIST_LEN],
                     nd['wind_cos'][t:t+HIST_LEN]], axis=1)      # (10,3,H,W)
 
    d_sin = np.empty((HIST_LEN, H, W), dtype=np.float32)
    d_cos = np.empty((HIST_LEN, H, W), dtype=np.float32)
    d_sin[:] = DIURNAL_SIN[:, None, None]
    d_cos[:] = DIURNAL_COS[:, None, None]
 
    lat_ch  = np.empty((HIST_LEN, H, W), dtype=np.float32)
    lon_ch  = np.empty((HIST_LEN, H, W), dtype=np.float32)
    pers_ch = np.empty((HIST_LEN, H, W), dtype=np.float32)
    lat_ch[:]  = STATIC_CH[0]
    lon_ch[:]  = STATIC_CH[1]
    pers_ch[:] = persistence
 
    # Channels: cpm25(1) warp(1) pers(1) met(8) emis(7) wind(3) div(1) dsin(1) dcos(1) lat(1) lon(1) = 25
    return np.concatenate([
        pm25_hist[:, None],    # 1
        warp_seq[:, None],     # 1
        pers_ch[:, None],      # 1
        met,                   # 8
        emis,                  # 7
        wind,                  # 3
        div_seq[:, None],      # 1
        d_sin[:, None],        # 1
        d_cos[:, None],        # 1
        lat_ch[:, None],       # 1
        lon_ch[:, None],       # 1
    ], axis=1).astype(np.float32)  # (10, 25, 140, 124)
 
 
# Quick shape verification
_out = extract_phase2(all_norm[MONTHS[0]], 0)
print(f"extract_phase2: {_out.shape}  (expected ({HIST_LEN}, {C_IN}, 140, 124))")
assert _out.shape == (HIST_LEN, C_IN, 140, 124), f"Shape mismatch: {_out.shape}"
del _out

extract_phase2: (10, 26, 140, 124)  (expected (10, 26, 140, 124))


In [8]:
#dataset
class PM25DatasetPhase2(Dataset):
    def __init__(self, norm_data, months, is_val=False, aug=True):
        self.nd     = norm_data
        self.is_val = is_val
        self.aug    = aug and not is_val
        self.samples = []
        for m in months:
            T     = norm_data[m]['cpm25'].shape[0]
            # Need HIST_LEN input + HORIZON target
            n_win = T - HIST_LEN - HORIZON + 1
            if is_val:
                self.samples += [(m, t) for t in range(max(0, n_win - VAL_HOURS), n_win)]
            else:
                self.samples += [(m, t) for t in range(max(0, n_win - VAL_HOURS))]
        print(f"  {'Val' if is_val else 'Train'}: {len(self.samples)} samples")
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, idx):
        m, t = self.samples[idx]
        x    = extract_phase2(self.nd[m], t)                             # (10, 25, H, W)
        y    = self.nd[m]['cpm25'][t+HIST_LEN:t+HIST_LEN+HORIZON].copy()  # (16, H, W)
        if self.aug:
            if np.random.random() < 0.5:
                x = x[:, :, :, ::-1].copy()
                y = y[:, :, ::-1].copy()
            if np.random.random() < 0.3:
                x = x[:, :, ::-1, :].copy()
                y = y[:, ::-1, :].copy()
        return torch.from_numpy(x), torch.from_numpy(y)
 
 
print("\nBuilding datasets")
train_ds = PM25DatasetPhase2(all_norm, MONTHS, is_val=False, aug=True)
val_ds   = PM25DatasetPhase2(all_norm, MONTHS, is_val=True,  aug=False)
 
NW = 2
PF = 2 if NW > 0 else None
train_ldr = DataLoader(train_ds, batch_size=BS, shuffle=True,
                       num_workers=NW, pin_memory=USE_AMP, drop_last=True,
                       persistent_workers=(NW > 0), prefetch_factor=PF)
val_ldr   = DataLoader(val_ds, batch_size=BS, shuffle=False,
                       num_workers=NW, pin_memory=USE_AMP,
                       persistent_workers=(NW > 0), prefetch_factor=PF)
 
print(f"  Train batches: {len(train_ldr)},  Val batches: {len(val_ldr)}")
x0, y0 = train_ds[0]
print(f"  x: {x0.shape}, y: {y0.shape}")
print(f"  Effective BS: {BS} x {ACCUM} = {BS*ACCUM}")


Building datasets
  Train: 2352 samples
  Val: 480 samples
  Train batches: 588,  Val batches: 120
  x: torch.Size([10, 26, 140, 124]), y: torch.Size([16, 140, 124])
  Effective BS: 4 x 4 = 16


**Model arch**

In [9]:
class ConvBN(nn.Module):
    def __init__(self, ic, oc, stride=1, k=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, oc, k, stride=stride, padding=k//2, bias=False),
            nn.BatchNorm2d(oc), nn.GELU(),
        )
    def forward(self, x): return self.net(x)
 
 
class ResConv(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.body = nn.Sequential(ConvBN(ic, oc), ConvBN(oc, oc))
        self.skip = nn.Conv2d(ic, oc, 1, bias=False) if ic != oc else nn.Identity()
    def forward(self, x): return self.body(x) + self.skip(x)
 
 
# class SpatialAttention(nn.Module):
#     def __init__(self, ch, reduction=8):
#         super().__init__()
#         mid = max(ch // reduction, 16)
#         self.q = nn.Conv2d(ch, mid, 1, bias=False)
#         self.k = nn.Conv2d(ch, mid, 1, bias=False)
#         self.v = nn.Conv2d(ch, ch,  1, bias=False)
#         self.scale = mid ** -0.5
 
#     def forward(self, x):
#         B, C, H, W = x.shape
#         q = self.q(x).reshape(B, -1, H*W).permute(0, 2, 1)   # B, HW, mid
#         k = self.k(x).reshape(B, -1, H*W)                     # B, mid, HW
#         v = self.v(x).reshape(B, C, H*W).permute(0, 2, 1)    # B, HW, C
#         attn = torch.softmax(torch.bmm(q, k) * self.scale, dim=-1)
#         out  = torch.bmm(attn, v).permute(0, 2, 1).reshape(B, C, H, W)
#         return x + out



class ChannelAttention(nn.Module):
    def __init__(self, ch, reduction=8):
        super().__init__()
        mid = max(ch // reduction, 16)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # squeeze: (B, C, H, W) -> (B, C, 1, 1)
            nn.Flatten(),              # (B, C)
            nn.Linear(ch, mid),
            nn.GELU(),
            nn.Linear(mid, ch),
            nn.Sigmoid(),

        )

    def forward(self, x):
        scale = self.se(x).reshape(x.shape[0], x.shape[1], 1, 1)
        return x * scale              # excitation: reweight channels
 
 
class FrameEncoderP2(nn.Module):
    def __init__(self, C_in=C_IN, base=BASE_CH, C_enc=C_ENC):
        super().__init__()
        self.level1 = nn.Sequential(
            ResConv(C_in, base),
            # SpatialAttention(base),
            ChannelAttention(base),
            ConvBN(base, base, stride=2),
        )
        self.level2 = nn.Sequential(
            ResConv(base, C_enc),
            # SpatialAttention(C_enc),
            ChannelAttention(C_enc),
            ConvBN(C_enc, C_enc, stride=2),
        )
 
    def forward(self, x):
        skip = self.level1(x)
        enc  = self.level2(skip)
        return enc, skip
 
 
class GatedBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.dropout  = dropout
        self.norm1    = nn.LayerNorm(d_model)
        self.qkv      = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj     = nn.Linear(d_model, d_model)
        self.gate1    = nn.Sequential(nn.Linear(d_model, d_model), nn.Sigmoid())
        self.norm2    = nn.LayerNorm(d_model)
        mid           = int(d_model * mlp_ratio)
        self.ffn      = nn.Sequential(
            nn.Linear(d_model, mid), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mid, d_model), nn.Dropout(dropout),
        )
        self.gate2 = nn.Sequential(nn.Linear(d_model, d_model), nn.Sigmoid())
 
    def forward(self, x):
        B, T, C = x.shape
        h       = self.norm1(x)
        q, k, v = self.qkv(h).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2,0,3,1,4).unbind(0)
        dp      = self.dropout if self.training else 0.0
        h       = F.scaled_dot_product_attention(q, k, v, dropout_p=dp)
        h       = self.proj(h.transpose(1,2).reshape(B, T, C))
        x       = x + h * self.gate1(x)
        x       = x + self.ffn(self.norm2(x)) * self.gate2(x)
        return x
 
 
class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.dropout  = dropout
        self.norm_q   = nn.LayerNorm(d_model)
        self.norm_kv  = nn.LayerNorm(d_model)
        self.q_proj   = nn.Linear(d_model, d_model, bias=False)
        self.k_proj   = nn.Linear(d_model, d_model, bias=False)
        self.v_proj   = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)
        self.gate     = nn.Sequential(nn.Linear(d_model, d_model), nn.Sigmoid())
        self.norm_ff  = nn.LayerNorm(d_model)
        mid = d_model * 4
        self.ffn = nn.Sequential(
            nn.Linear(d_model, mid), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mid, d_model), nn.Dropout(dropout),
        )
 
    def forward(self, query, context):
        # query:   (B, T_out, C)  — future horizon slots
        # context: (B, T_in,  C)  — historical encoded features
        B, Tq, C = query.shape
        q = self.q_proj(self.norm_q(query)).reshape(B, Tq, self.n_heads, self.head_dim).transpose(1,2)
        k = self.k_proj(self.norm_kv(context)).reshape(B, -1, self.n_heads, self.head_dim).transpose(1,2)
        v = self.v_proj(context).reshape(B, -1, self.n_heads, self.head_dim).transpose(1,2)
        dp  = self.dropout if self.training else 0.0
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp)
        out = self.out_proj(out.transpose(1,2).reshape(B, Tq, C))
        query = query + out * self.gate(query)
        query = query + self.ffn(self.norm_ff(query))
        return query
 
 
class TemporalTranslatorP2(nn.Module):
    """
    History encoder (self-attn) + Future projector (cross-attn); it prroduces HORIZON future feature maps from HIST_LEN inputs.
    """
    def __init__(self, C_enc=C_ENC, T_in=HIST_LEN, T_out=HORIZON,
                 n_heads=N_HEADS, n_blocks=N_BLOCKS, dropout=DROPOUT):
        super().__init__()
        self.T_out   = T_out
        self.C_enc   = C_enc
        self.pos_enc = nn.Parameter(torch.zeros(1, T_in, C_enc))
        nn.init.trunc_normal_(self.pos_enc, std=0.02)
        # Future query slots — learnable
        self.future_queries = nn.Parameter(torch.zeros(1, T_out, C_enc))
        nn.init.trunc_normal_(self.future_queries, std=0.02)
 
        self.self_blocks  = nn.ModuleList([
            GatedBlock(C_enc, n_heads, dropout=dropout) for _ in range(n_blocks // 2)
        ])
        self.cross_blocks = nn.ModuleList([
            CrossAttentionBlock(C_enc, n_heads, dropout=dropout) for _ in range(n_blocks // 2)
        ])
        self.norm = nn.LayerNorm(C_enc)
 
    def forward(self, x):
        # x: (B, T_in, C, Hp, Wp)
        B, T, C, Hp, Wp = x.shape
        # Flatten spatial for temporal attention
        xf = x.permute(0, 3, 4, 1, 2).reshape(B * Hp * Wp, T, C) + self.pos_enc
 
        # Self-attention on history
        for blk in self.self_blocks:
            xf = blk(xf)
        history = self.norm(xf)  # (B*Hp*Wp, T_in, C)
 
        # Cross-attention: future queries attend to history
        fq = self.future_queries.expand(B * Hp * Wp, -1, -1)
        for blk in self.cross_blocks:
            fq = blk(fq, history)
 
        # fq: (B*Hp*Wp, T_out, C) -> (B, T_out, C, Hp, Wp)
        return fq.reshape(B, Hp, Wp, self.T_out, C).permute(0, 3, 4, 1, 2)
 
 
class FrameDecoderP2(nn.Module):
    def __init__(self, C_enc=C_ENC, base=BASE_CH):
        super().__init__()
        self.up1    = nn.ConvTranspose2d(C_enc, base, 2, stride=2)
        self.fuse   = ResConv(base * 2, base)
        # self.sa     = SpatialAttention(base)
        self.ca     = ChannelAttention(base)
        self.up2    = nn.ConvTranspose2d(base, base // 2, 2, stride=2)
        self.refine = ResConv(base // 2, base // 2)
        self.head   = nn.Conv2d(base // 2, 1, 1)
        # Residual persistence head to help episode prediction
        self.res_head = nn.Conv2d(base // 2, 1, 1)
 
    def forward(self, enc, skip, persistence):
        B_T = enc.shape[0]
        B   = skip.shape[0]
        T   = B_T // B
        H   = persistence.shape[-2]   # original 140
        W   = persistence.shape[-1]   # original 124

        skip_exp = skip.unsqueeze(1).expand(-1, T, -1, -1, -1).reshape(B_T, *skip.shape[1:])
        x    = self.fuse(torch.cat([self.up1(enc), skip_exp], dim=1))
        x    = self.ca(x)
        x    = self.refine(self.up2(x))
        main = self.head(x)
        res  = self.res_head(x)

        # Crop padding before adding persistence
        main = main[:, :, PAD_H:PAD_H+H, PAD_W:PAD_W+W]
        res  = res[:,  :, PAD_H:PAD_H+H, PAD_W:PAD_W+W]

        pers_exp = persistence.unsqueeze(1).expand(
            -1, T, -1, -1, -1
        ).reshape(B_T, 1, H, W)

        return main + 0.3 * res + 0.1 * pers_exp
 
 
class PM25ForecasterPhase2(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder    = FrameEncoderP2()
        self.translator = TemporalTranslatorP2()
        self.decoder    = FrameDecoderP2()
 
    def forward(self, x):
        # x: (B, T_in, C, H, W)
        B, T, C, H, W = x.shape
        xp = F.pad(x.reshape(B*T, C, H, W),
                   (PAD_W, PAD_W, PAD_H, PAD_H), mode='reflect')
        enc, skip = self.encoder(xp)
        # Hp, Wp    = enc.shape[-2], enc.shape[-1]
        enc_seq   = enc.reshape(B, T, *enc.shape[1:])
        skip_avg  = skip.reshape(B, T, *skip.shape[1:]).mean(dim=1)
 
        # Persistence: channel 0 = pm25, last timestep
        persistence = x[:, -1, 0:1, :, :]  # (B, 1, H, W)
 
        out   = self.translator(enc_seq)                    # (B, T_out, C_enc, Hp, Wp)
        T_out = out.shape[1]
        pred  = self.decoder(
            out.reshape(B*T_out, *out.shape[2:]), skip_avg, persistence
        )
        # Crop padding: pred shape (B*T_out, 1, H+2P, W+2P)
        # pred = pred[:, 0, PAD_H:PAD_H+H, PAD_W:PAD_W+W]
        return pred[:,0].reshape(B, T_out, H, W)
 
 
# Instantiate and verify
model = PM25ForecasterPhase2().to(DEVICE)
n_p   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nPM25ForecasterPhase2 params: {n_p:,}")
 
with torch.no_grad():
    _x = torch.randn(2, HIST_LEN, C_IN, 140, 124, device=DEVICE)
    _o = model(_x)
    assert _o.shape == (2, HORIZON, 140, 124), f"Bad output shape: {_o.shape}"
    print(f"Forward pass OK: {_x.shape} -> {_o.shape}")
    del _x, _o
 
if DEVICE == 'cuda':
    print(f"  GPU mem after init: {torch.cuda.memory_allocated()/1024**2:.0f} MB")
    torch.cuda.empty_cache()
 
if hasattr(torch, 'compile') and DEVICE == 'cuda' and props.major >= 8:
    try:
        model = torch.compile(model, mode='reduce-overhead')
        print("torch.compile enabled")
    except Exception as e:
        print(f"torch.compile skipped: {e}")
else:
    print("torch.compile skipped")


PM25ForecasterPhase2 params: 1,550,290
Forward pass OK: torch.Size([2, 10, 26, 140, 124]) -> torch.Size([2, 16, 140, 124])
  GPU mem after init: 15 MB
torch.compile skipped


**Loss func**

In [10]:
class SMAPELoss(nn.Module):
    def forward(self, pred, target):
        p   = pred.float()
        t   = target.float()
        num = torch.abs(p - t)
        den = (torch.abs(p) + torch.abs(t)) / 2.0 + 1e-8
        return (num / den).mean()
 
 # SMAPE computed only on grid points above the episode threshold.
class EpisodeSMAPELoss(nn.Module):
    def __init__(self, threshold_norm):
        super().__init__()
        self.thresh = threshold_norm
 
    def forward(self, pred, target):
        p    = pred.float()
        t    = target.float()
        mask = (t > self.thresh).float()
        if mask.sum() < 1:
            return torch.tensor(0.0, device=pred.device)
        num  = torch.abs(p - t) * mask
        den  = ((torch.abs(p) + torch.abs(t)) / 2.0 + 1e-8) * mask
        return num.sum() / (den.sum() + 1e-8)
 
 
class EpisodeCorrelationLoss(nn.Module):
    
    def __init__(self, threshold_norm):
        super().__init__()
        self.thresh = threshold_norm
 
    def forward(self, pred, target):
        p    = pred.float().reshape(-1)
        t    = target.float().reshape(-1)
        mask = (t > self.thresh)
        if mask.sum() < 2:
            return torch.tensor(0.0, device=pred.device)
        p_ep = p[mask]
        t_ep = t[mask]
        pm   = p_ep - p_ep.mean()
        tm   = t_ep - t_ep.mean()
        corr = (pm * tm).sum() / (
            torch.sqrt((pm**2).sum() + 1e-8) *
            torch.sqrt((tm**2).sum() + 1e-8)
        )
        return 1.0 - corr
 
 
class SSIMLoss(nn.Module):
    def __init__(self, window_size=7, sigma=1.5):
        super().__init__()
        self.ws  = window_size
        self.pad = window_size // 2
        g        = torch.exp(-torch.arange(window_size).float()
                             .sub_(window_size//2).pow_(2).div_(2 * sigma**2))
        g        = g / g.sum()
        kernel   = g.outer(g).unsqueeze(0).unsqueeze(0)
        self.register_buffer('kernel', kernel)
 
    def forward(self, pred, target):
        p  = pred.float().reshape(-1, 1, pred.shape[-2], pred.shape[-1])
        t  = target.float().reshape(-1, 1, target.shape[-2], target.shape[-1])
        k  = self.kernel
        mu_p   = F.conv2d(p, k, padding=self.pad)
        mu_t   = F.conv2d(t, k, padding=self.pad)
        mu_p2  = mu_p * mu_p;  mu_t2 = mu_t * mu_t;  mu_pt = mu_p * mu_t
        sig_p2 = F.conv2d(p * p, k, padding=self.pad) - mu_p2
        sig_t2 = F.conv2d(t * t, k, padding=self.pad) - mu_t2
        sig_pt = F.conv2d(p * t, k, padding=self.pad) - mu_pt
        C1, C2 = 0.01**2, 0.03**2
        num    = (2*mu_pt + C1) * (2*sig_pt + C2)
        den    = (mu_p2 + mu_t2 + C1) * (sig_p2 + sig_t2 + C2)
        return 1.0 - (num / (den + 1e-8)).mean()
 
 
class TemporalGradientLoss(nn.Module):
    def forward(self, pred, target):
        p = pred.float();  t = target.float()
        return F.mse_loss(p[:, 1:] - p[:, :-1], t[:, 1:] - t[:, :-1])
 
 
class Phase2CompositeLoss(nn.Module):
    def __init__(self,
                 w_smape    = 0.30,
                 w_ep_smape = 0.25,
                 w_ep_corr  = 0.20,
                 w_ssim     = 0.10,
                 w_tgrad    = 0.08,
                 w_huber    = 0.07,
                 threshold_norm = 0.0):
        super().__init__()
        self.w_smape    = w_smape
        self.w_ep_smape = w_ep_smape
        self.w_ep_corr  = w_ep_corr
        self.w_ssim     = w_ssim
        self.w_tgrad    = w_tgrad
        self.w_huber    = w_huber
 
        self.smape    = SMAPELoss()
        self.ep_smape = EpisodeSMAPELoss(threshold_norm)
        self.ep_corr  = EpisodeCorrelationLoss(threshold_norm)
        self.ssim     = SSIMLoss()
        self.tgrad    = TemporalGradientLoss()
        self.huber    = nn.HuberLoss(delta=1.0, reduction='mean')
 
    def forward(self, pred, target):
        p = pred.float();  t = target.float()
        loss  = self.w_smape    * self.smape(p, t)
        loss  = loss + self.w_ep_smape * self.ep_smape(p, t)
        loss  = loss + self.w_ep_corr  * self.ep_corr(p, t)
        loss  = loss + self.w_ssim     * self.ssim(p, t)
        loss  = loss + self.w_tgrad    * self.tgrad(p, t)
        loss  = loss + self.w_huber    * self.huber(p, t)
        return loss
 
 
class RMSELoss(nn.Module):
    def forward(self, pred, target):
        return torch.sqrt(F.mse_loss(pred.float(), target.float()) + 1e-8)
 
 
thresh_norm = normalizer.episode_threshold_norm
print(f"\nEpisode threshold (normalised): {thresh_norm:.4f}")
criterion = Phase2CompositeLoss(threshold_norm=thresh_norm)
print("CompositeLoss defined.")
 


Episode threshold (normalised): 0.9473
CompositeLoss defined.


In [11]:
# EMA 

class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.model  = model
        self.decay  = decay
        self.shadow = {n: p.data.clone()
                       for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}
 
    @torch.no_grad()
    def update(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n] = self.decay * self.shadow[n] + (1-self.decay) * p.data
 
    def apply_shadow(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.backup[n] = p.data.clone()
                p.data.copy_(self.shadow[n])
 
    def restore(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad and n in self.backup:
                p.data.copy_(self.backup[n])
        self.backup = {}

In [12]:
#checkpoint utils

def save_checkpoint(path, model, optimizer, epoch, val_loss, norm_stats, tag=''):
    torch.save({
        'epoch': epoch, 'val_loss': val_loss,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'normalizer_stats': norm_stats,
    }, path)
    if tag:
        print(f"  Saved {tag}: {path.name}")

**Training**

In [13]:
def train_model(model, train_ldr, val_ldr, criterion,
                model_name='p2', epochs=EPOCHS, patience=PATIENCE,
                resume_path=None):
 
    optimizer = AdamW(model.parameters(), lr=LR_PEAK, weight_decay=WD)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=2, eta_min=5e-7)
    scaler    = torch.amp.GradScaler('cuda') if USE_SCALER else None
    ema       = EMA(model, decay=EMA_DECAY)
    val_crit  = RMSELoss()
    criterion.to(DEVICE)
 
    best_val   = float('inf')
    patience_c = 0
    start_ep   = 1
 
    if resume_path and Path(resume_path).exists():
        ck = torch.load(resume_path, map_location='cpu')
        model.load_state_dict(ck['model'])
        model.to(DEVICE)
        optimizer.load_state_dict(ck['optimizer'])
        start_ep  = ck['epoch'] + 1
        best_val  = ck['val_loss']
        print(f"  Resumed from epoch {ck['epoch']}, val={ck['val_loss']:.4f}")
        del ck
 
    print(f"\n{'_'*65}")
    print(f"TRAINING {model_name.upper()} — {epochs} ep | "
          f"BS={BS}x{ACCUM}={BS*ACCUM} | LR={LR_PEAK} | patience={patience}")
 
    for epoch in range(start_ep, epochs + 1):
        t0 = time.time()
 
        # Linear warmup
        if epoch <= WARMUP_EP:
            lr_scale = epoch / WARMUP_EP
            for pg in optimizer.param_groups:
                pg['lr'] = LR_PEAK * lr_scale
 
        model.train()
        train_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
 
        for step, (xb, yb) in enumerate(train_ldr):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
 
            if USE_AMP:
                with torch.amp.autocast('cuda', dtype=DTYPE):
                    pred = model(xb)
                loss = criterion(pred, yb) / ACCUM
            else:
                pred = model(xb)
                loss = criterion(pred, yb) / ACCUM
 
            if USE_SCALER:
                scaler.scale(loss).backward()
                if (step + 1) % ACCUM == 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer); scaler.update()
                    ema.update()
                    optimizer.zero_grad(set_to_none=True)
            else:
                loss.backward()
                if (step + 1) % ACCUM == 0:
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    ema.update()
                    optimizer.zero_grad(set_to_none=True)
 
            train_loss += loss.item() * ACCUM
 
        train_loss /= len(train_ldr)
 
        # Validate with EMA weights
        ema.apply_shadow()
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_ldr:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                if USE_AMP:
                    with torch.amp.autocast('cuda', dtype=DTYPE):
                        p = model(xb)
                else:
                    p = model(xb)
                val_loss += val_crit(p, yb).item()
        val_loss /= len(val_ldr)
        ema.restore()
 
        if epoch > WARMUP_EP:
            scheduler.step(epoch - WARMUP_EP)
 
        dt      = time.time() - t0
        is_best = val_loss < best_val
 
        if is_best:
            best_val   = val_loss
            patience_c = 0
            ema.apply_shadow()
            save_checkpoint(
                CKPT / f'best_{model_name}.pth', model, optimizer,
                epoch, val_loss, normalizer.stats,
                tag=f'BEST {model_name}'
            )
            ema.restore()
        else:
            patience_c += 1
 
        save_checkpoint(CKPT / f'latest_{model_name}.pth', model, optimizer,
                        epoch, val_loss, normalizer.stats)
        if epoch % CKPT_EVERY == 0:
            save_checkpoint(
                CKPT / f'{model_name}_ep{epoch:03d}.pth', model, optimizer,
                epoch, val_loss, normalizer.stats,
                tag=f'periodic {model_name} e{epoch}'
            )
 
        mem_str = (f" | GPU:{torch.cuda.memory_reserved()/1024**3:.1f}GB"
                   if DEVICE == 'cuda' else "")
        marker  = ' *' if is_best else ''
        lr_now  = optimizer.param_groups[0]['lr']
        print(f"Ep {epoch:3d}/{epochs} | tr={train_loss:.4f} val={val_loss:.4f} "
              f"lr={lr_now:.1e} {dt:.0f}s{mem_str}{marker}")
 
        if patience_c >= patience:
            print(f"\nEarly stop ep {epoch}. Best val: {best_val:.4f}")
            break
 
    est_rmse = best_val * normalizer.stats['cpm25']['std']
    print(f"\n {model_name} done. Best val RMSE (norm): {best_val:.4f} "
          f"≈ {est_rmse:.1f} µg/m³")
    return best_val, ema
 
 
# train 
best_val, ema = train_model(
    model, train_ldr, val_ldr, criterion,
    model_name='p2',
    epochs=EPOCHS,
    patience=PATIENCE,
    resume_path=CKPT / 'latest_p2.pth',
)


_________________________________________________________________
TRAINING P2 — 80 ep | BS=4x4=16 | LR=0.0002 | patience=20
  Saved BEST p2: best_p2.pth
Ep   1/80 | tr=0.4454 val=0.8231 lr=4.0e-05 236s | GPU:5.1GB *
  Saved BEST p2: best_p2.pth
Ep   2/80 | tr=0.3360 val=0.8027 lr=8.0e-05 221s | GPU:5.1GB *
  Saved BEST p2: best_p2.pth
Ep   3/80 | tr=0.3006 val=0.7811 lr=1.2e-04 222s | GPU:5.1GB *
  Saved BEST p2: best_p2.pth
Ep   4/80 | tr=0.2723 val=0.7664 lr=1.6e-04 221s | GPU:5.1GB *
  Saved BEST p2: best_p2.pth
Ep   5/80 | tr=0.2597 val=0.7563 lr=2.0e-04 222s | GPU:5.1GB *
  Saved BEST p2: best_p2.pth
Ep   6/80 | tr=0.2461 val=0.7331 lr=2.0e-04 223s | GPU:5.1GB *
  Saved BEST p2: best_p2.pth
Ep   7/80 | tr=0.2389 val=0.7252 lr=2.0e-04 221s | GPU:5.1GB *
  Saved BEST p2: best_p2.pth
Ep   8/80 | tr=0.2313 val=0.7059 lr=2.0e-04 221s | GPU:5.1GB *
  Saved BEST p2: best_p2.pth
Ep   9/80 | tr=0.2281 val=0.6852 lr=1.9e-04 222s | GPU:5.1GB *
  Saved BEST p2: best_p2.pth
  Saved periodic p

# Inference

In [14]:
del train_ldr, val_ldr, train_ds, val_ds, criterion
gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()
 
# Load best EMA checkpoint
ck = torch.load(CKPT / 'best_p2.pth', map_location='cpu')
model.load_state_dict(ck['model'])
model.to(DEVICE).eval()
print(f"\nLoaded best p2 (epoch {ck['epoch']}, val={ck['val_loss']:.4f})")
del ck
 
# Memory-map test data
print("\n Memory-mapping test data")
test_mmap = {}
for feat in ALL_FEATURES:
    test_mmap[feat] = np.load(TEST_IN / f"{feat}.npy", mmap_mode='r')
N_TEST = test_mmap['cpm25'].shape[0]
print(f"  cpm25 shape: {test_mmap['cpm25'].shape}")
print(f"  N_TEST     : {N_TEST}")
 
 
def _n(arr, feat):
    m = np.float32(normalizer.stats[feat]['mean'])
    s = np.float32(normalizer.stats[feat]['std'])
    return (np.asarray(arr, dtype=np.float32) - m) / s
 
 
def extract_test_phase2(i):
    """
    10-hour lookback from test_in; o/p shape: (HIST_LEN, C_IN, 140, 124)
    """
    H, W = 140, 124
 
    pm25_hist = _n(test_mmap['cpm25'][i], 'cpm25')   # (10, H, W)
    u_seq_raw = np.asarray(test_mmap['u10'][i], dtype=np.float32)
    v_seq_raw = np.asarray(test_mmap['v10'][i], dtype=np.float32)
    u_seq_n   = _n(u_seq_raw, 'u10')
    v_seq_n   = _n(v_seq_raw, 'v10')
 
    warp_seq  = compute_wind_warp_hist(pm25_hist, u_seq_n, v_seq_n)
    div_seq   = compute_divergence_hist(u_seq_n, v_seq_n)
    persistence = pm25_hist[-1]
 
    ws_raw = np.sqrt(u_seq_raw**2 + v_seq_raw**2).astype(np.float32)
    wd_raw = np.arctan2(v_seq_raw, u_seq_raw + 1e-8)
    ws_n   = _n(ws_raw, 'wind_speed')
    sin_n  = np.sin(wd_raw).astype(np.float32)
    cos_n  = np.cos(wd_raw).astype(np.float32)
 
    out = np.empty((HIST_LEN, C_IN, H, W), dtype=np.float32)
    for ts in range(HIST_LEN):
        c = 0
        out[ts, c] = pm25_hist[ts];  c += 1
        out[ts, c] = warp_seq[ts];   c += 1
        out[ts, c] = persistence;    c += 1
        for var in MET_VARS:
            out[ts, c] = _n(test_mmap[var][i, ts], var);  c += 1
        for var in EMIS_VARS:
            out[ts, c] = _n(test_mmap[var][i, ts], var);  c += 1
        out[ts, c] = ws_n[ts];    c += 1
        out[ts, c] = sin_n[ts];   c += 1
        out[ts, c] = cos_n[ts];   c += 1
        out[ts, c] = div_seq[ts]; c += 1
        out[ts, c] = DIURNAL_SIN[ts]; c += 1
        out[ts, c] = DIURNAL_COS[ts]; c += 1
        out[ts, c] = STATIC_CH[0];    c += 1
        out[ts, c] = STATIC_CH[1];    c += 1
    return out
 
 
@torch.no_grad()
def infer_batch(batch_np):
    b = torch.from_numpy(batch_np).to(DEVICE, non_blocking=True)
    if USE_AMP:
        with torch.amp.autocast('cuda', dtype=DTYPE):
            p = model(b)
    else:
        p = model(b)
    return p.float().cpu().numpy()
 
# TTA: original + horizontal flip + vertical flip + both flips
INFER_BS = 8
print(f"\nInference + 4x TTA (batch={INFER_BS})")
preds_raw = []
 
for bs in range(0, N_TEST, INFER_BS):
    be      = min(bs + INFER_BS, N_TEST)
    batch_x = np.stack([extract_test_phase2(i) for i in range(bs, be)])
 
    p_orig  = infer_batch(batch_x)
    p_hflip = infer_batch(batch_x[:, :, :, :, ::-1].copy())[:, :, :, ::-1]
    p_vflip = infer_batch(batch_x[:, :, :, ::-1, :].copy())[:, :, ::-1, :]
    p_both  = infer_batch(batch_x[:, :, :, ::-1, ::-1].copy())[:, :, ::-1, ::-1]
 
    preds_raw.append((p_orig + p_hflip + p_vflip + p_both) / 4.0)
 
    del batch_x, p_orig, p_hflip, p_vflip, p_both
    if be % 100 == 0 or be == N_TEST:
        print(f"  {be}/{N_TEST}")
 
preds = np.concatenate(preds_raw, axis=0)
del preds_raw
gc.collect()
 
# Denormalize and clip
preds = normalizer.denorm_pm25(preds)
preds = np.clip(preds, 0, None)
 
# Shape check: must be (N_TEST, 140, 124, 16)
preds = preds.transpose(0, 2, 3, 1).astype(np.float32)
assert preds.shape == (N_TEST, 140, 124, 16), f"Wrong shape: {preds.shape}"
 
# Save
out_file = OUTPUT / 'preds.npy'
np.save(out_file, preds)
 
print(f"SAVED : {out_file}")
print(f"Shape : {preds.shape}")
print(f"Mean  : {preds.mean():.2f} µg/m³")
print(f"Range : [{preds.min():.2f}, {preds.max():.2f}] µg/m³")
print(f"Size  : {out_file.stat().st_size / 1024**2:.1f} MB")
print(f"\nBest val RMSE (norm) : {best_val:.4f}")
print(f"                     ≈ {best_val * normalizer.stats['cpm25']['std']:.1f} µg/m³")



Loaded best p2 (epoch 73, val=0.2802)

 Memory-mapping test data
  cpm25 shape: (218, 10, 140, 124)
  N_TEST     : 218

Inference + 4x TTA (batch=8)
  200/218
  218/218
SAVED : /kaggle/working/preds.npy
Shape : (218, 140, 124, 16)
Mean  : 35.09 µg/m³
Range : [0.00, 950.25] µg/m³
Size  : 231.0 MB

Best val RMSE (norm) : 0.2802
                     ≈ 14.6 µg/m³
